# Labor forecast workflow conversion

Pandas translation of the Alteryx labor forecast, needs, and headcount actuals flows. Input workbooks are read by filename beside `currentlaborpath`; dashboard tables are written to SQL Server.

## Current Labor Forecast input

Read `Data - Forecast & Opps` from the workbook in `.env` key `currentlaborpath`; keep its filename.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
current_labor_path = Path(os.environ["currentlaborpath"])
current_labor_raw = pd.read_excel(
    current_labor_path,
    sheet_name="Data - Forecast & Opps",
    header=0,
)
current_labor_raw["FileName"] = current_labor_path.name
print(len(current_labor_raw), "current labor forecast rows loaded")

## Current GLC and months

Trim `GLC`; transpose month columns into rows, excluding `Labor Forecast Total`.

In [ ]:
current_labor_raw["GLC"] = current_labor_raw["GLC"].astype("string").str.strip()

key_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
]
missing_keys = [column for column in key_columns if column not in current_labor_raw.columns]
if missing_keys:
    raise KeyError(f"Missing current forecast key columns: {missing_keys}")

value_columns = [
    column for column in current_labor_raw.columns
    if column not in key_columns and column != "Labor Forecast Total"
]
current_labor = current_labor_raw.melt(
    id_vars=key_columns,
    value_vars=value_columns,
    var_name="Name",
    value_name="Value",
)
print(len(current_labor), "transposed rows from", len(value_columns), "data columns")

## Current month flag

`DoesHaveMonth` checks whether `Name` contains a digit so the forecast total is not treated as a month. Trim surrounding whitespace from `Segment`.

In [ ]:
current_labor["DoesHaveMonth"] = (
    current_labor["Name"].astype("string").str.contains(r"\d+", regex=True, na=False)
)
current_labor["Segment"] = current_labor["Segment"].astype("string").str.strip().str.slice(0, 255)
current_labor[["Name", "Value", "DoesHaveMonth", "Segment"]].head()

## Current month and category

Remove `Hours` from the month/year for date conversion; split `Name` on spaces and use its third part as `Category` for monthly hours. Rename `Value` to `NEW Hours`, remove temporary fields, and keep non-null `Type`.

In [ ]:
# RegEx (96) and DateTime (97)
current_labor["RegExOut1"] = current_labor["Name"].astype("string").str.extract(
    r"(\w+ \w+)", expand=False
)
current_labor["DateTime_Out"] = pd.to_datetime(
    current_labor["RegExOut1"].str.replace(
        r"^([A-Za-z]{3})[A-Za-z]+(?= \d{2}$)", r"\1", regex=True
    ),
    format="%b %y", errors="coerce"
)

# Select (103) and Text To Columns (318)
current_labor = current_labor.rename(columns={"Value": "NEW Hours"})
current_labor["NEW Hours"] = pd.to_numeric(current_labor["NEW Hours"], errors="coerce")
name_parts = current_labor["Name"].astype("string").str.split(" ", n=2, expand=True)
third_word = name_parts[2] if 2 in name_parts.columns else pd.Series(
    pd.NA, index=current_labor.index, dtype="string"
)

# Formula (100): Alteryx uses -1 for a true RegEx_Match.
current_labor["Category"] = current_labor["Name"].astype("string").where(
    ~current_labor["DoesHaveMonth"], third_word
)

# Select (101) and true output of Filter (102)
current_labor_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
    "NEW Hours", "RegExOut1", "DateTime_Out", "Category",
]
current_labor = current_labor.loc[
    current_labor["Type"].notna(), current_labor_columns
].copy()
current_labor.head()

## Prior Labor Forecast input and months

Read `Labor Forecast - Prior.xlsx` beside the current workbook, trim `GLC`, and transpose September 2026–August 2027 hours. Exclude `Labor Forecast Total`; warn about missing months.

In [ ]:
import warnings

prior_labor_path = current_labor_path.with_name("Labor Forecast - Prior.xlsx")
prior_labor_raw = pd.read_excel(
    prior_labor_path,
    sheet_name="Data - Forecast & Opps",
    header=0,
)
prior_labor_raw["FileName"] = prior_labor_path.name
prior_labor_raw["GLC"] = prior_labor_raw["GLC"].astype("string").str.strip().str.slice(0, 255)

prior_key_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
]
missing_prior_keys = [
    column for column in prior_key_columns if column not in prior_labor_raw.columns
]
if missing_prior_keys:
    raise KeyError(f"Missing prior forecast key columns: {missing_prior_keys}")

prior_month_columns = [
    "Sep 26 Hours", "Oct 26 Hours", "Nov 26 Hours", "Dec 26 Hours",
    "Jan 27 Hours", "Feb 27 Hours", "Mar 27 Hours", "Apr 27 Hours",
    "May 27 Hours", "June 27 Hours", "July 27 Hours", "Aug 27 Hours",
]
missing_prior_months = [
    column for column in prior_month_columns if column not in prior_labor_raw.columns
]
if missing_prior_months:
    warnings.warn(f"Prior forecast month columns missing: {missing_prior_months}")
present_prior_months = [
    column for column in prior_month_columns if column in prior_labor_raw.columns
]
prior_labor = prior_labor_raw.melt(
    id_vars=prior_key_columns,
    value_vars=present_prior_months,
    var_name="Name",
    value_name="Value",
)
print(len(prior_labor), "prior forecast rows from", len(present_prior_months), "month columns")


## Prior month and category

Repeat the month parsing and whitespace cleanup, rename `Value` to `Hours`, then keep non-null `Type`.

In [ ]:
# Formula (50), RegEx (51), and DateTime (52)
prior_labor["DoesHaveMonth"] = (
    prior_labor["Name"].astype("string").str.contains(r"\d+", regex=True, na=False)
)
prior_labor["Segment"] = prior_labor["Segment"].astype("string").str.strip().str.slice(0, 255)
prior_labor["RegExOut1"] = prior_labor["Name"].astype("string").str.extract(
    r"(\w+ \w+)", expand=False
)
prior_labor["DateTime_Out"] = pd.to_datetime(
    prior_labor["RegExOut1"].str.replace(
        r"^([A-Za-z]{3})[A-Za-z]+(?= \d{2}$)", r"\1", regex=True
    ),
    format="%b %y", errors="coerce"
)

# Select (58), Text To Columns (54), and Formula (55)
prior_labor = prior_labor.rename(columns={"Value": "Hours"})
prior_labor["Hours"] = pd.to_numeric(prior_labor["Hours"], errors="coerce")
prior_name_parts = prior_labor["Name"].astype("string").str.split(" ", n=2, expand=True)
prior_third_word = (
    prior_name_parts[2] if 2 in prior_name_parts.columns
    else pd.Series(pd.NA, index=prior_labor.index, dtype="string")
)
prior_labor["Category"] = prior_labor["Name"].astype("string").where(
    ~prior_labor["DoesHaveMonth"], prior_third_word
)

# Select (56) and the true output of Filter (57)
prior_labor_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
    "Hours", "RegExOut1", "DateTime_Out", "Category",
]
prior_labor = prior_labor.loc[
    prior_labor["Type"].notna(), prior_labor_columns
].copy()
prior_labor.head()


## Joining Prior and Current

Join on `Program`, `GLC`, month/year (`RegExOut1`), `Type`, and `Firm/FO-SS/New`. Keep matched rows plus unmatched rows whose `Category` is `Hours`; the latter removes total values without a match. Union by field name. The prior forecast retains its `Hours` name, and matched prior fields carry `Right_` prefixes.

In [ ]:
# Alteryx CHECK checkpoint: prior Hours summed by GLC and Program.
prior_hours_checkpoint = prior_labor.groupby(
    ["GLC", "Program"], dropna=False, as_index=False
)["Hours"].sum()

join_fields = ["Program", "GLC", "RegExOut1", "Type", "Firm/FO-SS/New"]
current_has_keys = current_labor[join_fields].notna().all(axis=1)
prior_has_keys = prior_labor[join_fields].notna().all(axis=1)

# Alteryx does not join null keys; keep those rows for the unmatched outputs.
comparison = current_labor.loc[current_has_keys].merge(
    prior_labor.loc[prior_has_keys].add_prefix("Right_"),
    how="outer",
    left_on=join_fields,
    right_on=[f"Right_{field}" for field in join_fields],
    indicator=True,
    sort=False,
)
left_fields = current_labor.columns.tolist()
right_fields = [f"Right_{field}" for field in prior_labor.columns]

current_unmatched = pd.concat(
    [
        comparison.loc[comparison["_merge"].eq("left_only"), left_fields],
        current_labor.loc[~current_has_keys],
    ],
    ignore_index=True,
)
prior_unmatched = pd.concat(
    [
        comparison.loc[comparison["_merge"].eq("right_only"), right_fields].rename(
            columns=lambda column: column.removeprefix("Right_")
        ),
        prior_labor.loc[~prior_has_keys],
    ],
    ignore_index=True,
)
current_unmatched_hours = current_unmatched.loc[
    current_unmatched["Category"].eq("Hours").fillna(False)
]
prior_unmatched_hours = prior_unmatched.loc[
    prior_unmatched["Category"].eq("Hours").fillna(False)
]
matched_forecasts = comparison.loc[comparison["_merge"].eq("both")].drop(
    columns="_merge"
)

combined_forecasts = pd.concat(
    [current_unmatched_hours, matched_forecasts, prior_unmatched_hours],
    ignore_index=True,
    sort=False,
)
print(
    len(matched_forecasts), "matched,",
    len(current_unmatched_hours), "current only,",
    len(prior_unmatched_hours), "prior only"
)
combined_forecasts.head()


## CHECK: prior hours after the join

Sum `Hours` by `GLC` and `Program` after the union. Alteryx's `IsNull([Sum_Hours])` true output identifies groups with no prior hours; retain both outputs for inspection.

In [ ]:
combined_hours_checkpoint = combined_forecasts.groupby(
    ["GLC", "Program"], dropna=False, as_index=False
)["Hours"].sum(min_count=1).rename(columns={"Hours": "Sum_Hours"})
null_prior_hours = combined_hours_checkpoint.loc[
    combined_hours_checkpoint["Sum_Hours"].isna()
].copy()
with_prior_hours = combined_hours_checkpoint.loc[
    combined_hours_checkpoint["Sum_Hours"].notna()
].copy()
print(len(null_prior_hours), "groups with null prior hours")


## Work Hours Per Month

Read the `Reports` sheet of `COBRA Calendar.xlsx`. Keep the `MD50CAL_NG` fiscal file and its date, hours, and month label fields. The Alteryx note calls this the fiscal hours per month source.

In [ ]:
calendar_path = current_labor_path.with_name("COBRA Calendar.xlsx")
calendar_raw = pd.read_excel(
    calendar_path, sheet_name="Reports", header=0
)
calendar_renames = {
    "Calendar File": "FISCFILE",
    "Date": "FSC_DATE",
    "Hours": "HOURS",
    "00 Label": "FIELD00",
}
missing_calendar_fields = [
    field for field in calendar_renames if field not in calendar_raw.columns
]
if missing_calendar_fields:
    raise KeyError(f"Missing COBRA calendar fields: {missing_calendar_fields}")
calendar = calendar_raw.rename(columns=calendar_renames)
calendar = calendar.loc[
    calendar["FISCFILE"].eq("MD50CAL_NG"),
    ["FSC_DATE", "HOURS", "FIELD00"],
].copy()
calendar.head()


## Calendar month and PTO

Convert `FIELD00` with Alteryx's custom `Mon yy` format into a month date. Keep the calendar date, work hours, `00 Label`, and parsed `DateTime_Out`. PTO values apply to each fiscal month.

In [ ]:
calendar["DateTime_Out"] = pd.to_datetime(
    calendar["FIELD00"].astype("string").str.strip(),
    format="%b %y",
    errors="coerce",
)
calendar = calendar.rename(columns={
    "FSC_DATE": "Date",
    "HOURS": "Work Hours in Month",
    "FIELD00": "00 Label",
})
calendar["Date"] = pd.to_datetime(calendar["Date"], errors="coerce")
calendar["Work Hours in Month"] = pd.to_numeric(
    calendar["Work Hours in Month"], errors="coerce"
)
calendar = calendar[
    ["Date", "Work Hours in Month", "00 Label", "DateTime_Out"]
].copy()

pto_path = current_labor_path.with_name("PTO Values.xlsx")
pto_raw = pd.read_excel(pto_path, sheet_name="PTO", header=0)
pto_columns = ["PTO Current", "PTO Prior"]
missing_pto_columns = [col for col in pto_columns if col not in pto_raw.columns]
if missing_pto_columns:
    raise KeyError(f"Missing PTO values: {missing_pto_columns}")
pto = pto_raw[pto_columns].copy()
for column in pto_columns:
    pto[column] = pd.to_numeric(pto[column], errors="coerce")
if len(pto) > 16:
    raise ValueError("Alteryx Append Fields allows at most 16 PTO records")
print(len(calendar), "calendar rows,", len(pto), "PTO records")

## Adjusted work hours

Append current and prior PTO rates to fiscal hours. The Alteryx notes calculate monthly hours after PTO as `Work Hours in Month × (1 − PTO)` for each period. Select 201 removes the PTO inputs before the date join.

In [ ]:
# Append Fields (199) is a Cartesian append of the PTO table.
work_hours = calendar.merge(pto, how="cross")
work_hours["NEW Adjust Work Hours"] = (
    work_hours["Work Hours in Month"] * (1 - work_hours["PTO Current"])
)
work_hours["Adjust Work Hours"] = (
    work_hours["Work Hours in Month"] * (1 - work_hours["PTO Prior"])
)
work_hours = work_hours[
    [
        "Date", "Work Hours in Month", "00 Label", "DateTime_Out",
        "NEW Adjust Work Hours", "Adjust Work Hours",
    ]
].copy()
work_hours.head()

## GLC details

Select the GLC details workbook, then join its `GLC`, element description, organization, and function to the combined forecast on `GLC` (Join 67).

In [ ]:
glc_details_path = current_labor_path.with_name("GLC Details.xlsx")
glc_details_raw = pd.read_excel(glc_details_path, sheet_name="GLC Details", header=0)
glc_fields = ["GLC", "Element Description", "Org", "Function"]
missing_glc_fields = [field for field in glc_fields if field not in glc_details_raw.columns]
if missing_glc_fields:
    raise KeyError(f"Missing GLC details fields: {missing_glc_fields}")
glc_details = glc_details_raw[glc_fields].copy()
glc_details["GLC"] = glc_details["GLC"].astype("string").str.strip()
forecast_with_glc = combined_forecasts.merge(
    glc_details,
    on="GLC",
    how="inner",
    validate="many_to_many",
)
print(len(forecast_with_glc), "rows matched to GLC details")

## Join forecast to work hours

Join the GLC-enriched forecast to Select 201 by `DateTime_Out`. Calculate direct equivalence using monthly work hours, as noted after Join 47.

In [ ]:
forecast_with_work_hours = forecast_with_glc.merge(
    work_hours,
    on="DateTime_Out",
    how="inner",
    suffixes=("", "_calendar"),
    validate="many_to_many",
)
forecast_with_work_hours["Direct Equivalence"] = (
    pd.to_numeric(forecast_with_work_hours["Hours"], errors="coerce")
    / forecast_with_work_hours["Work Hours in Month"]
)
forecast_with_work_hours["NEW DE"] = (
    pd.to_numeric(forecast_with_work_hours["NEW Hours"], errors="coerce")
    / forecast_with_work_hours["Work Hours in Month"]
)
print(len(forecast_with_work_hours), "forecast rows matched to fiscal month hours")
forecast_with_work_hours.head()

## DLY and OT budgets

Read current and prior DLY/OT values from adjacent workbooks. Join on `GLC Code`, keeping the current budgets as `NEW Budget DLY` and `NEW Budget OT`; combine the matched and unmatched GLC rows by field name (Join 185 and Union 186).

In [ ]:
dly_current_path = current_labor_path.with_name("DLY&OT Values - Current.xlsx")
dly_prior_path = current_labor_path.with_name("DLY&OT Values - Prior.xlsx")
dly_current_raw = pd.read_excel(dly_current_path, sheet_name="DLY&OT", header=0)
dly_prior_raw = pd.read_excel(dly_prior_path, sheet_name="DLY&OT", header=0)

budget_fields = ["GLC Code", "Budget DLY", "Budget OT"]
for label, frame in [("current", dly_current_raw), ("prior", dly_prior_raw)]:
    missing = [field for field in budget_fields if field not in frame.columns]
    if missing:
        raise KeyError(f"Missing {label} DLY/OT fields: {missing}")
dly_current_fields = [
    "GLC Code", "GLC Description", "Organization Number",
    "Organization Description", "Ex / Nex", "OH Pool",
    "Budget DLY", "Budget OT",
]
missing_current_details = [field for field in dly_current_fields if field not in dly_current_raw.columns]
if missing_current_details:
    raise KeyError(f"Missing current DLY/OT detail fields: {missing_current_details}")
dly_current = dly_current_raw[dly_current_fields].rename(columns={
    "Budget DLY": "NEW Budget DLY",
    "Budget OT": "NEW Budget OT",
}).copy()
dly_prior = dly_prior_raw[budget_fields].copy()
for frame in (dly_current, dly_prior):
    frame["GLC Code"] = frame["GLC Code"].astype("string").str.strip()
for frame, fields in [
    (dly_current, ["NEW Budget DLY", "NEW Budget OT"]),
    (dly_prior, ["Budget DLY", "Budget OT"]),
]:
    for field in fields:
        frame[field] = pd.to_numeric(frame[field], errors="coerce")

dly_budgets = dly_current.merge(
    dly_prior, on="GLC Code", how="outer", validate="many_to_many"
)
print(len(dly_budgets), "combined DLY/OT GLC rows")

## Budget adjusted FTEs

Join the enriched forecast to DLY/OT by `GLC` = `GLC Code` (Join 75). Calculate prior and current FTEs using their respective PTO adjusted work hours, DLY, and OT budgets.

In [ ]:
forecast_with_budgets = forecast_with_work_hours.merge(
    dly_budgets,
    left_on="GLC",
    right_on="GLC Code",
    how="inner",
    validate="many_to_many",
)
prior_denominator = (
    forecast_with_budgets["Adjust Work Hours"]
    * forecast_with_budgets["Budget DLY"]
    * (1 + forecast_with_budgets["Budget OT"])
)
new_denominator = (
    forecast_with_budgets["NEW Adjust Work Hours"]
    * forecast_with_budgets["NEW Budget DLY"]
    * (1 + forecast_with_budgets["NEW Budget OT"])
)
forecast_with_budgets["FTEs"] = (
    forecast_with_budgets["Hours"] / prior_denominator.where(prior_denominator.ne(0))
)
forecast_with_budgets["NEW FTEs"] = (
    forecast_with_budgets["NEW Hours"] / new_denominator.where(new_denominator.ne(0))
)
forecast_with_budgets.head()

## Needs: current forecast

For Actuals vs Needs, Join 221 adds GLC descriptions and function to the current forecast. Join 203 adds fiscal work hours by month. Join 222 adds current DLY/OT budget values by GLC. The Alteryx notes calculate direct equivalence and current FTEs from these fields.

In [ ]:
needs_glc = current_labor.merge(
    glc_details, on="GLC", how="inner", validate="many_to_many"
)
needs_with_hours = needs_glc.merge(
    work_hours[
        ["DateTime_Out", "Work Hours in Month",
         "NEW Adjust Work Hours", "Adjust Work Hours"]
    ],
    on="DateTime_Out", how="inner", validate="many_to_many",
)
needs_with_budgets = needs_with_hours.merge(
    dly_current,
    left_on="GLC", right_on="GLC Code",
    how="inner", validate="many_to_many",
    suffixes=("", "_budget"),
)
# Formula 215 uses a FixedDecimal scale of 2.
needs_with_budgets["Direct Equivalence"] = (
    needs_with_budgets["NEW Hours"] / needs_with_budgets["Work Hours in Month"]
).round(2)
needs_denominator = (
    needs_with_budgets["NEW Adjust Work Hours"]
    * needs_with_budgets["NEW Budget DLY"]
    * (1 + needs_with_budgets["NEW Budget OT"])
)
needs_with_budgets["FTEs"] = (
    needs_with_budgets["NEW Hours"]
    / needs_denominator.where(needs_denominator.ne(0))
)
needs_with_budgets.head()

## Needs: monthly sum and average

Sum FTEs by type, forecast month, GLC code, org, and function. Average those monthly sums across months for each type and organization; Select 237 renames the average to `Sum_FTEs`.

In [ ]:
monthly_needs = needs_with_budgets.groupby(
    ["Type", "RegExOut1", "GLC Code", "Org", "Function"],
    dropna=False, as_index=False,
)["FTEs"].sum(min_count=1).rename(columns={"FTEs": "Sum_FTEs"})

needs = monthly_needs.groupby(
    ["Type", "GLC Code", "Org", "Function"],
    dropna=False, as_index=False,
)["Sum_FTEs"].mean()
needs.head()

## Headcount Actuals: employee demographics

Filter 258 selects the employee demographics workbook from the original input directory. Read its `Employee Demographics` sheet, keep active regular full-time and part-time employees (Filters 261/263), and union them by field name (Union 264).

In [ ]:
demographics_files = list(
    current_labor_path.parent.glob("*Employee Demographics.xlsx")
)
if len(demographics_files) != 1:
    raise FileNotFoundError(
        "Expected exactly one Employee Demographics.xlsx workbook in "
        f"{current_labor_path.parent}; found {len(demographics_files)}"
    )
demographics = pd.read_excel(
    demographics_files[0], sheet_name="Employee Demographics", header=0
)
active_regular = (
    demographics["EMPL_STATUS"].eq("A")
    & demographics["REG_TEMP"].eq("R")
)
full_time = demographics.loc[
    active_regular & demographics["FULL_PART_TIME"].eq("F")
].copy()
part_time = demographics.loc[
    active_regular & demographics["FULL_PART_TIME"].eq("P")
].copy()
actual_employees = pd.concat(
    [full_time, part_time], ignore_index=True, sort=False
)
actual_employees["Org"] = (
    actual_employees["ATK_GL_DEPT"].astype("string").str[-3:]
)
print(len(actual_employees), "active regular employees")


## Headcount Actuals: GLC mapping and count

Join 233 maps `ATK_GENL_LABOR_CAT` to GLC details. Formula 252 expresses each employee's `STD_HOURS` as a fraction of a 40-hour week.

In [ ]:
actuals_with_glc = actual_employees.merge(
    glc_details,
    left_on="ATK_GENL_LABOR_CAT",
    right_on="GLC",
    how="inner",
    suffixes=("", "_glc"),
    validate="many_to_many",
)
actuals_with_glc["Right_Org"] = actuals_with_glc["Org_glc"]
actuals_with_glc["COUNT"] = (
    pd.to_numeric(actuals_with_glc["STD_HOURS"], errors="coerce") / 40
)
actuals_with_glc.head()


## Headcount Actuals vs Needs

Sum `COUNT` by labor category, Org, and Function (Summarize 253), then join average FTE need on GLC Code, Org, and Function (Join 239). Opportunity rows have zero actual headcount so a person mapped to both forecast types is not counted twice. Keep unmatched rows from either side (Union 235).

In [ ]:
headcount_actuals = (
    actuals_with_glc.groupby(
        ["ATK_GENL_LABOR_CAT", "Org", "Function"],
        dropna=False, as_index=False,
    )["COUNT"]
    .sum(min_count=1)
    .rename(columns={
        "ATK_GENL_LABOR_CAT": "GLC Code",
        "COUNT": "Count",
    })
)
headcount_vs_actuals = needs.merge(
    headcount_actuals,
    on=["GLC Code", "Org", "Function"],
    how="outer",
    validate="many_to_many",
)
headcount_vs_actuals.loc[
    headcount_vs_actuals["Type"].eq("Opportunity"), "Count"
] = 0
headcount_vs_actuals = headcount_vs_actuals.rename(columns={
    "Sum_FTEs": "FTE Need",
    "Count": "Headcount Actuals",
})
headcount_vs_actuals = headcount_vs_actuals[
    ["Type", "GLC Code", "Org", "Function", "FTE Need", "Headcount Actuals"]
]
headcount_vs_actuals.head()


## Final comparison rows

Select 80 removes temporary right-side join fields and file names, keeping the forecast, GLC details, work hours, budgets, and FTEs. Append the file-name/last-updated summary and run timestamp for the dashboard.

In [ ]:
output_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "NEW Hours",
    "RegExOut1", "DateTime_Out", "Category", "Hours",
    "Element Description", "Org", "Function", "Work Hours in Month",
    "NEW Adjust Work Hours", "Adjust Work Hours", "Direct Equivalence",
    "NEW DE", "GLC Code", "GLC Description", "Organization Number",
    "Organization Description", "Ex / Nex", "OH Pool",
    "NEW Budget DLY", "NEW Budget OT", "Budget DLY", "Budget OT",
    "FTEs", "NEW FTEs",
]
missing_output_columns = [
    column for column in output_columns if column not in forecast_with_budgets.columns
]
if missing_output_columns:
    raise KeyError(f"Missing Select 80 output fields: {missing_output_columns}")
final_forecast = forecast_with_budgets[output_columns].copy()

# File Names and Last Updated: summarize the input workbooks used by this run.
input_paths = [
    current_labor_path, prior_labor_path, glc_details_path,
    dly_current_path, dly_prior_path, calendar_path, pto_path,
]
file_updates = [
    f"{path.name}: {pd.Timestamp(path.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M:%S')}"
    for path in input_paths
]
final_forecast["Concat_File Name & Last Time Updated"] = "; ".join(file_updates)
final_forecast["TimeStamp"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
final_forecast.head()

## Write dashboard table

Write `LaborForecastQuarterCompareFinal_dev` with pandas after reviewing the output above. Use SQL Server credentials from `.env`.

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={os.environ['server']};"
    f"DATABASE={os.environ['database']};"
    "ENCRYPT=yes;"
    f"UID={os.environ['username']};"
    f"PWD={os.environ['password']};"
    "TrustServerCertificate=YES"
)
engine = create_engine(URL.create(
    "mssql+pyodbc", query={"odbc_connect": connection_string}
))
# Output Data (85): Overwrite Table (Drop), mapped by field name.
final_forecast.to_sql(
    "LaborForecastQuarterCompareFinal_dev",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False,
    chunksize=10000,
)
print(len(final_forecast), "rows written")

## Write HeadcountVSActuals

Write `HeadcountVSActuals_dev` beside the dashboard headcount table with the need and actuals rows.

In [ ]:
headcount_vs_actuals.to_sql(
    "HeadcountVSActuals_dev",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False,
    chunksize=10000,
)
print(len(headcount_vs_actuals), "headcount rows written")
